<a href="https://colab.research.google.com/github/zencolab/WhatDreamsCost-ComfyUI/blob/main/Qwen-Image-Edit-2511.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================
# Cell 1: 安装 ComfyUI + Manager
# ==========================================
import os
from pathlib import Path

print("=== 🚀 安装 ComfyUI 原生版，不使用 GGUF ===")

os.environ["OPENCV_IO_ENABLE_OPENEXR"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

COMFY_DIR = Path("/content/ComfyUI")

!apt-get -y update -qq
!apt-get -y install -qq aria2 git wget ffmpeg

!pip install -q -U pip
!pip install -q -U huggingface_hub hf_xet

if not COMFY_DIR.exists():
    !git clone https://github.com/comfyanonymous/ComfyUI /content/ComfyUI

%cd /content/ComfyUI

!pip install -q -r requirements.txt

# 安装 ComfyUI Manager
MANAGER_DIR = COMFY_DIR / "custom_nodes" / "ComfyUI-Manager"

if not MANAGER_DIR.exists():
    !git clone https://github.com/ltdrdata/ComfyUI-Manager.git /content/ComfyUI/custom_nodes/ComfyUI-Manager

print("✅ ComfyUI + Manager 安装完成")

In [ ]:
# ==========================================
# Cell 2: 下载 Qwen-Image-Edit-2511 原模型 + Multiple-Angles LoRA + Lightning LoRA + Workflow
# ==========================================
import os
import shutil
from pathlib import Path
from huggingface_hub import hf_hub_download, login

print("=== 📦 下载模型文件 ===")

COMFY_DIR = Path("/content/ComfyUI")

DIFFUSION_DIR = COMFY_DIR / "models" / "diffusion_models"
TEXT_ENCODER_DIR = COMFY_DIR / "models" / "text_encoders"
VAE_DIR = COMFY_DIR / "models" / "vae"
LORA_DIR = COMFY_DIR / "models" / "loras"
WORKFLOW_DIR = Path("/content/workflows")
HF_DOWNLOAD_DIR = Path("/content/hf_downloads")

for p in [
    DIFFUSION_DIR,
    TEXT_ENCODER_DIR,
    VAE_DIR,
    LORA_DIR,
    WORKFLOW_DIR,
    HF_DOWNLOAD_DIR,
]:
    p.mkdir(parents=True, exist_ok=True)

# 可选：使用 Colab userdata 里的 HF_TOKEN，避免下载限速
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = None

if HF_TOKEN:
    login(token=HF_TOKEN)
    print("✅ HF_TOKEN 登录成功")
else:
    print("⚠️ 未设置 HF_TOKEN，使用匿名下载，可能会限速")


def download_and_copy(repo_id, filename, dst_dir, dst_name=None):
    """
    从 Hugging Face 下载单个文件，并复制到 ComfyUI 对应目录。
    repo_id: Hugging Face 仓库名
    filename: 仓库内文件路径
    dst_dir: 目标目录
    dst_name: 目标文件名，默认使用 filename 的 basename
    """
    print("\n----------------------------------------")
    print("Repo:", repo_id)
    print("File:", filename)

    src = hf_hub_download(
        repo_id=repo_id,
        filename=filename,
        local_dir=str(HF_DOWNLOAD_DIR),
        local_dir_use_symlinks=False,
    )

    if dst_name is None:
        dst_name = Path(filename).name

    dst = Path(dst_dir) / dst_name

    if not dst.exists():
        shutil.copy2(src, dst)
        print("✅ 已复制到:", dst)
    else:
        print("✅ 已存在，跳过复制:", dst)

    return dst


# ------------------------------
# 1. Qwen-Image-Edit-2511 BF16 原生 ComfyUI 主模型
#    放到 ComfyUI/models/diffusion_models
# ------------------------------
main_model_dst = download_and_copy(
    repo_id="Comfy-Org/Qwen-Image-Edit_ComfyUI",
    filename="split_files/diffusion_models/qwen_image_edit_2511_bf16.safetensors",
    dst_dir=DIFFUSION_DIR,
    dst_name="qwen_image_edit_2511_bf16.safetensors",
)

# ------------------------------
# 2. Text Encoder
#    放到 ComfyUI/models/text_encoders
# ------------------------------
text_encoder_dst = download_and_copy(
    repo_id="f5aiteam/CLIP",
    filename="qwen_2.5_vl_7b_fp8_scaled.safetensors",
    dst_dir=TEXT_ENCODER_DIR,
    dst_name="qwen_2.5_vl_7b_fp8_scaled.safetensors",
)

# ------------------------------
# 3. VAE
#    放到 ComfyUI/models/vae
# ------------------------------
vae_dst = download_and_copy(
    repo_id="f5aiteam/VAE",
    filename="qwen_image_vae.safetensors",
    dst_dir=VAE_DIR,
    dst_name="qwen_image_vae.safetensors",
)

# ------------------------------
# 4. Multiple-Angles LoRA
#    放到 ComfyUI/models/loras
# ------------------------------
angle_lora_dst = download_and_copy(
    repo_id="fal/Qwen-Image-Edit-2511-Multiple-Angles-LoRA",
    filename="qwen-image-edit-2511-multiple-angles-lora.safetensors",
    dst_dir=LORA_DIR,
    dst_name="qwen-image-edit-2511-multiple-angles-lora.safetensors",
)

# ------------------------------
# 5. Lightning 4steps LoRA
#    红框缺失的就是这个
#    放到 ComfyUI/models/loras
# ------------------------------
lightning_lora_dst = download_and_copy(
    repo_id="lightx2v/Qwen-Image-Edit-2511-Lightning",
    filename="Qwen-Image-Edit-2511-Lightning-4steps-V1.0-bf16.safetensors",
    dst_dir=LORA_DIR,
    dst_name="Qwen-Image-Edit-2511-Lightning-4steps-V1.0-bf16.safetensors",
)

# ------------------------------
# 6. Multiple-Angles ComfyUI workflow
#    放到 /content/workflows
# ------------------------------
angle_workflow_dst = download_and_copy(
    repo_id="fal/Qwen-Image-Edit-2511-Multiple-Angles-LoRA",
    filename="comfyui-workflow-multiple-angles.json",
    dst_dir=WORKFLOW_DIR,
    dst_name="comfyui-workflow-multiple-angles.json",
)

# ------------------------------
# 7. 官方 Qwen-Image-Edit-2511 Native workflow，备用
# ------------------------------
official_workflow_dst = WORKFLOW_DIR / "image_qwen_image_edit_2511.json"

if not official_workflow_dst.exists():
    !wget -q -O {official_workflow_dst} https://raw.githubusercontent.com/Comfy-Org/workflow_templates/refs/heads/main/templates/image_qwen_image_edit_2511.json
    print("✅ 官方 Native Workflow 已下载:", official_workflow_dst)
else:
    print("✅ 官方 Native Workflow 已存在:", official_workflow_dst)


print("\n=== ✅ 下载完成，模型目录检查 ===")

print("\n--- diffusion_models ---")
!find /content/ComfyUI/models/diffusion_models -maxdepth 1 -type f -printf "%f\n"

print("\n--- text_encoders ---")
!find /content/ComfyUI/models/text_encoders -maxdepth 1 -type f -printf "%f\n"

print("\n--- vae ---")
!find /content/ComfyUI/models/vae -maxdepth 1 -type f -printf "%f\n"

print("\n--- loras ---")
!find /content/ComfyUI/models/loras -maxdepth 1 -type f -printf "%f\n"

print("\n--- workflows ---")
!find /content/workflows -maxdepth 1 -type f -printf "%f\n"

print("\n=== 💽 磁盘状态 ===")
!df -h /content

print("\n=== 📦 目录占用 ===")
!du -sh /content/ComfyUI/models/* 2>/dev/null || true
!du -sh /content/workflows 2>/dev/null || true

In [ ]:
# ==========================================
# Cell 3: FRP 内网穿透，可选
# ==========================================
import os
import subprocess
from pathlib import Path

print("=== 🌐 配置 FRP，可选 ===")

try:
    from google.colab import userdata
    VPS_IP = userdata.get("VPS_IP")
    FRP_TOKEN = userdata.get("FRP_TOKEN")
except Exception:
    VPS_IP = None
    FRP_TOKEN = None

if not VPS_IP or not FRP_TOKEN:
    print("⚠️ 未设置 VPS_IP / FRP_TOKEN，跳过 FRP")
else:
    FRP_DIR = Path("/content/frp_0.56.0_linux_amd64")

    if not FRP_DIR.exists():
        subprocess.run(
            "wget -qO- https://github.com/fatedier/frp/releases/download/v0.56.0/frp_0.56.0_linux_amd64.tar.gz | tar -xz -C /content",
            shell=True,
            check=True,
        )

    conf = f'''
serverAddr = "{VPS_IP}"
serverPort = 7000
auth.token = "{FRP_TOKEN}"

[[proxies]]
name = "comfyui_web_colab"
type = "tcp"
localIP = "127.0.0.1"
localPort = 8188
remotePort = 8090
'''

    (FRP_DIR / "frpc.toml").write_text(conf.strip(), encoding="utf-8")

    print("✅ FRP 配置完成")
    print("frpc.toml:")
    print((FRP_DIR / "frpc.toml").read_text())

In [ ]:
# ==========================================
# Cell 4: 启动 ComfyUI
# ==========================================
import os
import subprocess
import threading
import time
import configparser
from pathlib import Path

COMFY_DIR = Path("/content/ComfyUI")
FRP_DIR = Path("/content/frp_0.56.0_linux_amd64")

os.environ["OPENCV_IO_ENABLE_OPENEXR"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

def keep_alive():
    while True:
        time.sleep(300)
        print("\n[Keep-Alive] Colab 保活中...")

threading.Thread(target=keep_alive, daemon=True).start()

# 启动 FRP
frpc = FRP_DIR / "frpc"
frpc_conf = FRP_DIR / "frpc.toml"

if frpc.exists() and frpc_conf.exists():
    def start_frpc():
        subprocess.run([str(frpc), "-c", str(frpc_conf)])

    threading.Thread(target=start_frpc, daemon=True).start()
    print("✅ FRP 已启动")
    print("👉 访问: http://cjp.usdream.dpdns.org:8090")
else:
    print("⚠️ 未启用 FRP")

# Manager private 模式，减少启动 Fetch 等待
manager_paths = [
    COMFY_DIR / "user" / "__manager" / "config.ini",
    COMFY_DIR / "user" / "default" / "ComfyUI-Manager" / "config.ini",
    COMFY_DIR / "custom_nodes" / "ComfyUI-Manager" / "config.ini",
]

for p in manager_paths:
    p.parent.mkdir(parents=True, exist_ok=True)
    cfg = configparser.ConfigParser()

    if p.exists():
        cfg.read(p)

    if "default" not in cfg:
        cfg["default"] = {}

    cfg["default"]["network_mode"] = "private"

    with open(p, "w") as f:
        cfg.write(f)

print("✅ Manager private 模式完成")

os.chdir(COMFY_DIR)
print("🚀 启动 ComfyUI...")

subprocess.run([
    "python",
    "main.py",
    "--listen",
    "127.0.0.1",
    "--port",
    "8188",
    "--dont-print-server",
])